# HackPsyAssistant — fine-tuning du scorer (Colab GPU)

**Avant de lancer :** `Exécution → Modifier le type d'exécution → T4 GPU`.

1. Exécute les cellules dans l'ordre.
2. La cellule *Upload* te demande 2 fichiers du projet : `scripts/train.py` et `data/synthetic.jsonl`.
3. À la fin, un fichier **`xlmr-fr.zip`** est téléchargé automatiquement.
4. De retour dans le projet : décompresse-le dans `models/` puis lance
   `HACKPSY_MODEL_DIR=models/xlmr-fr uv run hackpsyassistant`.

Durée totale attendue sur T4 : **~15–25 min**.

In [ ]:
# 1) GPU présent ?
!nvidia-smi -L
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
# 2) Dépendances (mêmes versions majeures que le projet : transformers v5)
!pip install -q -U "transformers>=5.23,<6" "datasets>=5" scikit-learn accelerate
import transformers; print('transformers', transformers.__version__)

In [ ]:
# 3) Upload de train.py et synthetic.jsonl (depuis ton projet local)
import os, shutil
from google.colab import files
os.makedirs('data', exist_ok=True)
need = [f for f in ('train.py', 'synthetic.jsonl') if not os.path.exists(f) and not os.path.exists(f'data/{f}')]
if need:
    print('Sélectionne :', need)
    up = files.upload()
    for name in up:
        if name.endswith('.jsonl'):
            shutil.move(name, 'data/synthetic.jsonl')
assert os.path.exists('train.py') and os.path.exists('data/synthetic.jsonl'), 'fichiers manquants'
print('OK — lignes synthetic:', sum(1 for _ in open('data/synthetic.jsonl')))

In [ ]:
# 4) Entraînement principal : XLM-R multilingue (~15 min sur T4)
!python train.py --model FacebookAI/xlm-roberta-base --out models/xlmr-fr

In [ ]:
# 5) (Optionnel, pour le tableau comparatif du jury) un 2e modèle plus petit
RUN_COMPARISON = False
if RUN_COMPARISON:
    !python train.py --model distilbert-base-multilingual-cased --out models/distil-fr

In [ ]:
# 6) Test rapide sur des phrases françaises avant de télécharger
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
tok = AutoTokenizer.from_pretrained('models/xlmr-fr')
model = AutoModelForSequenceClassification.from_pretrained('models/xlmr-fr').eval()
tests = [
    "Je n'arrive plus à dormir, je me réveille à 3h chaque nuit",
    "J'ai annulé avec mes amis, je préfère rester seul",
    "Je me sens un fardeau pour tout le monde",
    "Super journée, j'ai bien avancé sur mon projet !",
]
enc = tok(tests, return_tensors='pt', padding=True, truncation=True)
with torch.no_grad():
    p = torch.sigmoid(model(**enc).logits)
labels = [model.config.id2label[i] for i in range(p.shape[1])]
for t, row in zip(tests, p):
    print(t[:50].ljust(52), {l: round(float(v), 2) for l, v in zip(labels, row)})

In [ ]:
# 7) Zip + téléchargement -> décompresser dans models/ du projet
import shutil
from google.colab import files
shutil.make_archive('xlmr-fr', 'zip', 'models/xlmr-fr')
files.download('xlmr-fr.zip')